# Classical Coherence Model Experiments

Leak-free classical baselines for ROCStories coherence checking. Run the setup cells first, then choose quick or full execution at the bottom.


In [1]:
import argparse
import json
import logging
import os
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


SEED = 42
SENT_COLS = [f"sentence{i}" for i in range(1, 6)]


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


@dataclass
class ExperimentConfig:
    data_path: str = "ROCStories_winter2017 - ROCStories_winter2017 (2).csv"
    output_dir: str = "outputs/experiments"
    test_size: float = 0.15
    val_size: float = 0.15
    neg_per_pos: int = 1
    seed: int = SEED


In [2]:
class LinearSVCWithScores(BaseEstimator, ClassifierMixin):
    """LinearSVC wrapper exposing probability-like scores for ROC/PR curves."""

    def __init__(self, C: float = 1.0, class_weight: Optional[str] = None, max_iter: int = 5000):
        self.C = C
        self.class_weight = class_weight
        self.max_iter = max_iter
        self.model = LinearSVC(C=C, class_weight=class_weight, max_iter=max_iter)

    def fit(self, X, y):
        self.model.fit(X, y)
        self.classes_ = self.model.classes_
        return self

    def predict(self, X):
        return self.model.predict(X)

    def decision_function(self, X):
        return self.model.decision_function(X)


In [3]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


In [4]:
def load_rocstories(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    missing = df[SENT_COLS].isnull().sum().sum()
    if missing:
        raise ValueError(f"Found {missing} missing sentence values; clean the CSV before training.")
    log.info("Loaded %s stories from %s", f"{len(df):,}", path)
    return df


In [5]:
def split_raw_stories(
    df: pd.DataFrame,
    test_size: float,
    val_size: float,
    seed: int,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Split before negative generation to avoid same-story leakage."""

    trainval_df, test_df = train_test_split(df, test_size=test_size, random_state=seed, shuffle=True)
    val_relative = val_size / (1 - test_size)
    train_df, val_df = train_test_split(trainval_df, test_size=val_relative, random_state=seed, shuffle=True)
    log.info(
        "Raw-story split | train=%s val=%s test=%s",
        f"{len(train_df):,}",
        f"{len(val_df):,}",
        f"{len(test_df):,}",
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


In [6]:
def make_negative(sentences: List[str], rng: random.Random, strategy: str) -> List[str]:
    if strategy == "shuffle":
        negative = sentences[:]
        while negative == sentences:
            rng.shuffle(negative)
        return negative

    if strategy == "adjacent_swap":
        negative = sentences[:]
        idx = rng.randrange(len(sentences) - 1)
        negative[idx], negative[idx + 1] = negative[idx + 1], negative[idx]
        return negative

    if strategy == "reverse":
        return list(reversed(sentences))

    raise ValueError(f"Unknown negative strategy: {strategy}")


In [7]:
def build_labeled_split(
    raw_df: pd.DataFrame,
    neg_per_pos: int,
    seed: int,
    negative_strategy: str,
) -> pd.DataFrame:
    rng = random.Random(seed)
    records = []

    for story_id, row in raw_df.iterrows():
        sentences = [row[c] for c in SENT_COLS]
        records.append(
            {
                "story_id": row.get("storyid", story_id),
                "story": " [SEP] ".join(sentences),
                "label": 1,
                "negative_strategy": "original",
            }
        )
        for _ in range(neg_per_pos):
            negative = make_negative(sentences, rng, negative_strategy)
            records.append(
                {
                    "story_id": row.get("storyid", story_id),
                    "story": " [SEP] ".join(negative),
                    "label": 0,
                    "negative_strategy": negative_strategy,
                }
            )

    return pd.DataFrame(records).sample(frac=1, random_state=seed).reset_index(drop=True)


In [8]:
def metric_dict(y_true: Iterable[int], y_pred: Iterable[int], y_score: Optional[Iterable[float]]) -> Dict[str, float]:
    y_true = np.asarray(list(y_true))
    y_pred = np.asarray(list(y_pred))
    row = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
    }
    if y_score is not None:
        y_score = np.asarray(list(y_score))
        row["roc_auc"] = roc_auc_score(y_true, y_score)
        row["pr_auc"] = average_precision_score(y_true, y_score)
    else:
        row["roc_auc"] = np.nan
        row["pr_auc"] = np.nan
    return row


In [9]:
def positive_scores(model, X) -> Optional[np.ndarray]:
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        return 1 / (1 + np.exp(-scores))
    return None


In [10]:
def classical_models(seed: int) -> Dict[str, Pipeline]:
    word_tfidf = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=120_000,
        sublinear_tf=True,
    )
    char_tfidf = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_features=120_000,
        sublinear_tf=True,
    )

    return {
        "majority_dummy": Pipeline(
            [
                ("tfidf", TfidfVectorizer(max_features=1000)),
                ("clf", DummyClassifier(strategy="most_frequent")),
            ]
        ),
        "stratified_dummy": Pipeline(
            [
                ("tfidf", TfidfVectorizer(max_features=1000)),
                ("clf", DummyClassifier(strategy="stratified", random_state=seed)),
            ]
        ),
        "word_tfidf_logreg": Pipeline(
            [
                ("tfidf", word_tfidf),
                ("clf", LogisticRegression(max_iter=2000, C=2.0, random_state=seed)),
            ]
        ),
        "char_tfidf_logreg": Pipeline(
            [
                ("tfidf", char_tfidf),
                ("clf", LogisticRegression(max_iter=2000, C=2.0, random_state=seed)),
            ]
        ),
        "word_tfidf_linear_svm": Pipeline(
            [
                ("tfidf", word_tfidf),
                ("clf", LinearSVCWithScores(C=1.0, max_iter=5000)),
            ]
        ),
    }


In [11]:
def run_classical_experiments(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    output_dir: Path,
    seed: int,
) -> pd.DataFrame:
    X_train, y_train = train_df["story"], train_df["label"]
    X_val, y_val = val_df["story"], val_df["label"]
    X_test, y_test = test_df["story"], test_df["label"]

    rows = []
    for name, model in classical_models(seed).items():
        log.info("Training %s", name)
        model.fit(X_train, y_train)

        val_pred = model.predict(X_val)
        val_score = positive_scores(model, X_val)
        test_pred = model.predict(X_test)
        test_score = positive_scores(model, X_test)

        row = {"model": name, "split_used_for_selection": "validation"}
        row.update({f"val_{k}": v for k, v in metric_dict(y_val, val_pred, val_score).items()})
        row.update({f"test_{k}": v for k, v in metric_dict(y_test, test_pred, test_score).items()})
        rows.append(row)

        model_dir = output_dir / name
        model_dir.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(confusion_matrix(y_test, test_pred)).to_csv(model_dir / "test_confusion_matrix.csv", index=False)
        with open(model_dir / "test_classification_report.txt", "w", encoding="utf-8") as f:
            f.write(
                classification_report(
                    y_test,
                    test_pred,
                    target_names=["Incoherent", "Coherent"],
                    zero_division=0,
                )
            )

    results = pd.DataFrame(rows).sort_values(["val_f1", "test_f1"], ascending=False)
    results.to_csv(output_dir / "classical_model_comparison.csv", index=False)
    return results


In [12]:
def save_splits(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame, output_dir: Path) -> None:
    split_dir = output_dir / "splits"
    split_dir.mkdir(parents=True, exist_ok=True)
    train_df.to_csv(split_dir / "train.csv", index=False)
    val_df.to_csv(split_dir / "val.csv", index=False)
    test_df.to_csv(split_dir / "test.csv", index=False)


## Run Classical Experiments

Set `QUICK = True` for a fast smoke test. Set it to `False` for final reported results.


In [13]:
DATA_PATH = "ROCStories_winter2017 - ROCStories_winter2017 (2).csv"
OUTPUT_DIR = "outputs/experiments"
NEGATIVE_STRATEGY = "shuffle"  # options: "shuffle", "adjacent_swap", "reverse"
NEG_PER_POS = 1
QUICK = True

cfg = ExperimentConfig(
    data_path=DATA_PATH,
    output_dir=OUTPUT_DIR,
    neg_per_pos=NEG_PER_POS,
)
set_seed(cfg.seed)
output_dir = Path(cfg.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

raw_df = load_rocstories(cfg.data_path)
if QUICK:
    raw_df = raw_df.sample(n=min(2000, len(raw_df)), random_state=cfg.seed).reset_index(drop=True)
    log.info("Quick mode enabled: using %s raw stories", f"{len(raw_df):,}")

train_raw, val_raw, test_raw = split_raw_stories(raw_df, cfg.test_size, cfg.val_size, cfg.seed)
train_df = build_labeled_split(train_raw, cfg.neg_per_pos, cfg.seed, NEGATIVE_STRATEGY)
val_df = build_labeled_split(val_raw, cfg.neg_per_pos, cfg.seed + 1, NEGATIVE_STRATEGY)
test_df = build_labeled_split(test_raw, cfg.neg_per_pos, cfg.seed + 2, NEGATIVE_STRATEGY)

save_splits(train_df, val_df, test_df, output_dir)
with open(output_dir / "experiment_config.json", "w", encoding="utf-8") as f:
    json.dump({**asdict(cfg), "negative_strategy": NEGATIVE_STRATEGY, "quick": QUICK}, f, indent=2)

classical_results = run_classical_experiments(train_df, val_df, test_df, output_dir, cfg.seed)
classical_results


18:18:38 [INFO] Loaded 52,665 stories from ROCStories_winter2017 - ROCStories_winter2017 (2).csv
18:18:38 [INFO] Quick mode enabled: using 2,000 raw stories
18:18:38 [INFO] Raw-story split | train=1,400 val=300 test=300
18:18:38 [INFO] Training majority_dummy
18:18:38 [INFO] Training stratified_dummy
18:18:38 [INFO] Training word_tfidf_logreg
18:18:39 [INFO] Training char_tfidf_logreg
18:18:40 [INFO] Training word_tfidf_linear_svm


,model,split_used_for_selection,val_accuracy,val_precision,val_recall,val_f1,val_mcc,val_roc_auc,val_pr_auc,test_accuracy,test_precision,test_recall,test_f1,test_mcc,test_roc_auc,test_pr_auc
4,word_tfidf_linear_svm,validation,0.605000,0.581818,0.746667,0.654015,0.218973,0.639589,0.633644,0.586667,0.566327,0.740000,0.641618,0.182108,0.639506,0.642617
2,word_tfidf_logreg,validation,0.596667,0.574359,0.746667,0.649275,0.202668,0.633778,0.625689,0.576667,0.557500,0.743333,0.637143,0.162635,0.631783,0.632889
1,stratified_dummy,validation,0.506667,0.506944,0.486667,0.496599,0.013344,0.506667,0.503380,0.506667,0.506944,0.486667,0.496599,0.013344,0.506667,0.503380
0,majority_dummy,validation,0.500000,0.000000,0.000000,0.000000,0.000000,0.500000,0.500000,0.500000,0.000000,0.000000,0.000000,0.000000,0.500000,0.500000
3,char_tfidf_logreg,validation,0.500000,0.000000,0.000000,0.000000,0.000000,0.500000,0.500000,0.500000,0.000000,0.000000,0.000000,0.000000,0.500000,0.500000
